# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [3]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [4]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="workplace-documents", #Give it a meaningful name,
    embedding=embeddings,
)

Elastic Cloud ID: ··········
Elastic Api Key: ··········
OpenAI API key: ··········


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [5]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [1]:
!pip install -U langchain-community

from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import json

def metadata_func(record: dict, metadata: dict) -> dict:
    """
    Populates the metadata dictionary with specified keys from the record.
    """
    metadata['name'] = record.get('name')
    metadata['summary'] = record.get('summary')
    metadata['url'] = record.get('url')
    metadata['category'] = record.get('category')
    metadata['updated_at'] = record.get('updated_at')

    return metadata

# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
print("Loading data from temp.json...")
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

# Define chunk size and chunk overlap as specified
CHUNK_SIZE = 800
CHUNK_OVERLAP = 400

print(f"Initializing text splitter with chunk_size={CHUNK_SIZE} and chunk_overlap={CHUNK_OVERLAP}...")
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

# Load and split the documents
docs = loader.load_and_split(text_splitter=text_splitter)

print(f"\nSuccessfully loaded {len(loader.load())} documents and split them into {len(docs)} passages.")
print("The first passage and its metadata are shown below:")
print("--------------------------------------------------")
print("Passage Content:")
print(f"'{docs[0].page_content[:250]}...'")  # Print a snippet for readability
print("\nPassage Metadata:")
print(docs[0].metadata)
print("--------------------------------------------------")

Loading data from temp.json...
Initializing text splitter with chunk_size=800 and chunk_overlap=400...

Successfully loaded 15 documents and split them into 15 passages.
The first passage and its metadata are shown below:
--------------------------------------------------
Passage Content:
'Effective: March 2020
Purpose

The purpose of this full-time work-from-home policy is to provide guidelines and support for employees to conduct their work remotely, ensuring the continuity and productivity of business operations during the COVID-19 ...'

Passage Metadata:
{'source': '/content/temp.json', 'seq_num': 1, 'name': 'Work From Home Policy', 'summary': 'This policy outlines the guidelines for full-time remote work, including eligibility, equipment and resources, workspace requirements, communication expectations, performance expectations, time tracking and overtime, confidentiality and data security, health and well-being, and policy reviews and updates. Employees are encouraged to direct a

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [5]:
import getpass
import json
import logging
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_elasticsearch import ElasticsearchStore
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers.multi_query import MultiQueryRetriever

# Set up logging to see the queries generated by the LLM
logging.basicConfig(level=logging.INFO)
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# Get credentials securely
ELASTIC_CLOUD_ID = getpass.getpass("Elastic Cloud ID:")
ELASTIC_API_KEY = getpass.getpass("Elastic Api Key:")
OPENAI_API_KEY = getpass.getpass("OpenAI API key:")

# Define metadata function
def metadata_func(record: dict, metadata: dict) -> dict:
    metadata['name'] = record.get('name')
    metadata['summary'] = record.get('summary')
    metadata['url'] = record.get('url')
    metadata['category'] = record.get('category')
    metadata['updated_at'] = record.get('updated_at')
    return metadata

# Define chunking parameters
CHUNK_SIZE = 800
CHUNK_OVERLAP = 400

# Load and split documents
print("Loading and splitting documents from temp.json...")
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
docs = loader.load_and_split(text_splitter=text_splitter)
print(f"Successfully loaded and split documents into {len(docs)} passages.")

# Initialize OpenAI embeddings and bulk import to Elasticsearch
print("\nInitializing Elasticsearch and OpenAI connections and performing bulk import...")
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# This line defines the 'vectorstore' object.
# The previous error occurred because this line was not executed before it was used.
vectorstore = ElasticsearchStore.from_documents(
    docs,
    embeddings,
    index_name="workplace-documents-split",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)
print("\nBulk import to Elasticsearch complete.")

# Initialize the LLM for query generation
llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

# Create the MultiQueryRetriever
print("Creating the MultiQueryRetriever...")
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)
print("Retriever created successfully.")

# Test the retriever with a sample query
question = "What is the new flexible working policy?"
print(f"\nTesting the retriever with the question: '{question}'")
retrieved_docs = retriever.invoke(question)

print("\n--- Retrieved Documents (unique) ---")
for doc in retrieved_docs:
    print(f"- {doc.page_content}")

Elastic Cloud ID:··········
Elastic Api Key:··········
OpenAI API key:··········
Loading and splitting documents from temp.json...
Successfully loaded and split documents into 15 passages.

Initializing Elasticsearch and OpenAI connections and performing bulk import...

Bulk import to Elasticsearch complete.
Creating the MultiQueryRetriever...
Retriever created successfully.

Testing the retriever with the question: 'What is the new flexible working policy?'


INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the updated flexible working policy?', '2. How has the working policy changed in terms of flexibility?', '3. What are the key features of the new flexible working policy?']



--- Retrieved Documents (unique) ---
- As we continue to prioritize the well-being of our employees, we are making a slight adjustment to our hybrid work policy. Starting May 1, 2023, employees will be required to work from the office three days a week, with two days designated for remote work. Please communicate with your supervisor and HR department to establish your updated in-office workdays.
- Starting May 2022, the company will be implementing a two-day in-office work requirement per week for all eligible employees. Please coordinate with your supervisor and HR department to schedule your in-office workdays while continuing to follow all safety protocols.
- Effective: March 2020
Purpose

The purpose of this full-time work-from-home policy is to provide guidelines and support for employees to conduct their work remotely, ensuring the continuity and productivity of business operations during the COVID-19 pandemic and beyond.
Scope

This policy applies to all employees who are elig

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [6]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible.

    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization of the company. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services in the North and South American markets. They work closely with other departments, such as marketing, product development, and customer support, to ensure the success of the company's products and services in these regions.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [10]:
!pip install langchain-google-genai

import getpass
import json
from urllib.request import urlopen
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_elasticsearch import ElasticsearchStore
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough

import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# Get credentials securely
ELASTIC_CLOUD_ID = getpass.getpass("Elastic Cloud ID:")
ELASTIC_API_KEY = getpass.getpass("Elastic Api Key:")
# This iteration uses Google's Gemini model
GOOGLE_API_KEY = getpass.getpass("Google API key:")
# We'll still use OpenAI for embeddings for simplicity, as per the original setup.
OPENAI_API_KEY = getpass.getpass("OpenAI API key:")

# Define metadata function
def metadata_func(record: dict, metadata: dict) -> dict:
    metadata['name'] = record.get('name')
    metadata['summary'] = record.get('summary')
    metadata['url'] = record.get('url')
    metadata['category'] = record.get('category')
    metadata['updated_at'] = record.get('updated_at')
    return metadata

# Define chunking parameters
CHUNK_SIZE = 800
CHUNK_OVERLAP = 400

# Load and split documents
print("Loading and splitting documents from temp.json...")
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
docs = loader.load_and_split(text_splitter=text_splitter)
print(f"Successfully loaded and split documents into {len(docs)} passages.")

# Initialize OpenAI embeddings and bulk import to Elasticsearch
print("\nInitializing Elasticsearch and embedding documents...")
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
vectorstore = ElasticsearchStore.from_documents(
    docs,
    embeddings,
    index_name="workplace-documents-gemini",  # New index for this iteration
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)
print("\nBulk import to Elasticsearch complete.")

# Initialize the LLM for query generation using Gemini
print("Creating the RAG chain with a Gemini LLM...")
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest", google_api_key=GOOGLE_API_KEY, temperature=0)
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# Define the prompts and the document combining function
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible.

    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)

def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)

# Build the RAG chain using Runnables
_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)
chain = _context | LLM_CONTEXT_PROMPT | llm

print("RAG chain built successfully.")

# Ask a question and get the final answer
question = "What is the policy regarding pets in the office?"
print(f"\nAsking the RAG chain the question: '{question}'")
ans = chain.invoke(question)

print("---- Final Answer ----")
print(ans.content)



Elastic Cloud ID:··········
Elastic Api Key:··········
Google API key:··········
OpenAI API key:··········
Loading and splitting documents from temp.json...
Successfully loaded and split documents into 15 passages.

Initializing Elasticsearch and embedding documents...

Bulk import to Elasticsearch complete.
Creating the RAG chain with a Gemini LLM...
RAG chain built successfully.

Asking the RAG chain the question: 'What is the policy regarding pets in the office?'


INFO:langchain.retrievers.multi_query:Generated queries: ['What are the rules and regulations concerning animals in the workplace?', 'Are pets allowed in the company building, and if so, under what conditions?', "What is the company's stance on allowing animals, specifically pets, on office premises?"]


---- Final Answer ----
The office pet policy outlines guidelines and procedures for bringing pets to work, aiming to create a positive and inclusive environment while ensuring everyone's safety and well-being.  The policy applies to dogs, cats, and other small, non-exotic animals, but requires prior approval from both the employee's supervisor and the HR department.

The approval process involves submitting a written request detailing the pet (breed, age, temperament), providing proof of vaccinations and licenses, and obtaining written consent from all coworkers sharing the workspace.  HR retains the right to deny or revoke approval based on circumstances or employee concerns.

Once approved, employees are fully responsible for their pet's behavior and well-being in the office.  Pets must be well-behaved, house-trained, and supervised at all times.  They must be leashed or in a secure enclosure outside the employee's immediate work area.  Owners are responsible for cleaning up after th

In [8]:
import getpass
import json
from urllib.request import urlopen
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_elasticsearch import ElasticsearchStore
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from typing import List

import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# Get credentials securely
ELASTIC_CLOUD_ID = getpass.getpass("Elastic Cloud ID:")
ELASTIC_API_KEY = getpass.getpass("Elastic Api Key:")
OPENAI_API_KEY = getpass.getpass("OpenAI API key:")

# Define metadata function
def metadata_func(record: dict, metadata: dict) -> dict:
    metadata['name'] = record.get('name')
    metadata['summary'] = record.get('summary')
    metadata['url'] = record.get('url')
    metadata['category'] = record.get('category')
    metadata['updated_at'] = record.get('updated_at')
    return metadata

# Define chunking parameters
CHUNK_SIZE = 800
CHUNK_OVERLAP = 400

# Load and split documents
print("Loading and splitting documents from temp.json...")
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
docs = loader.load_and_split(text_splitter=text_splitter)
print(f"Successfully loaded and split documents into {len(docs)} passages.")

# Initialize OpenAI embeddings and bulk import to Elasticsearch
print("\nInitializing Elasticsearch and embedding documents...")
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
vectorstore = ElasticsearchStore.from_documents(
    docs,
    embeddings,
    index_name="workplace-documents-filtered", # New index for this iteration
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)
print("\nBulk import to Elasticsearch complete.")

# Initialize the LLM and the MultiQueryRetriever
print("Creating the MultiQueryRetriever...")
llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# This new function filters documents based on a keyword list.
def filter_documents(docs: List, keywords: List[str]) -> List:
    filtered_docs = []
    for doc in docs:
        if any(keyword.lower() in doc.page_content.lower() for keyword in keywords):
            filtered_docs.append(doc)
    return filtered_docs

# Define the prompts and the document combining function
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible.

    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)

def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)

# Build the RAG chain using Runnables, now with a filtering step
print("Building the RAG chain with post-retrieval filtering...")
_context = RunnableParallel(
    # The retriever now pipes its results into a filter function
    context=retriever | (lambda docs: filter_documents(docs, ["work-from-home", "remote", "hybrid", "flexible"])) | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

print("RAG chain built successfully.")

# Ask a question and get the final answer
question = "What is the new flexible working policy?"
print(f"\nAsking the RAG chain the question: '{question}'")
ans = chain.invoke(question)

print("---- Final Answer ----")
print(ans)


Elastic Cloud ID:··········
Elastic Api Key:··········
OpenAI API key:··········
Loading and splitting documents from temp.json...
Successfully loaded and split documents into 15 passages.

Initializing Elasticsearch and embedding documents...

Bulk import to Elasticsearch complete.
Creating the MultiQueryRetriever...
Building the RAG chain with post-retrieval filtering...
RAG chain built successfully.

Asking the RAG chain the question: 'What is the new flexible working policy?'


INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the updated flexible working policy?', '2. How has the working policy changed in terms of flexibility?', '3. What are the key features of the new flexible working policy?']


---- Final Answer ----
The new flexible working policy, effective May 1, 2023, requires employees to work from the office three days a week and allows for two days of remote work. This policy is being implemented to prioritize the well-being of employees and ensure the continuity and productivity of business operations. Employees must communicate with their supervisor and HR department to establish their updated in-office workdays.


Yes, I can confidently say I've mastered the concepts of the Multi-Query Retriever and the broader RAG pipeline. The two iterations above demonstrate this in a few key ways:

    Modularity: The first file shows that the LLM is a modular component that can be easily swapped out for a different provider (like Google's Gemini) without changing the core retrieval logic. This is a fundamental concept in building flexible and scalable RAG systems.

    Pipeline Enhancement: The second file demonstrates how to build on the basic pipeline by adding a post-retrieval filtering step. This is a crucial technique for improving the quality of the documents sent to the LLM, leading to more accurate and concise answers.

The lab provided a solid foundation, and these creative iterations show how to extend it for more advanced, real-world applications.